In [1]:
from torch.utils.data import DataLoader, TensorDataset

from gensim.models import Word2Vec
from model         import GenerateModel
from metrics       import eval_model, compare_metric
import matplotlib.pyplot as plt
import numpy             as np
import torch
import os

device     = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu'); print(f'Deivce: {device}')
model_path = os.path.join(os.getcwd(),'Model','model.tar')

Deivce: mps


In [2]:
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')

Y_train = np.load('Y_train.npy')
Y_test  = np.load('Y_test.npy')

X_train = torch.from_numpy(X_train).long()
Y_train = torch.from_numpy(Y_train).type(torch.float32)
X_test  = torch.from_numpy(X_test) .long()
Y_test  = torch.from_numpy(Y_test) .type(torch.float32) 

train_loader = DataLoader(TensorDataset(X_train,Y_train), batch_size = 16,shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test ,Y_test) , batch_size = 64,shuffle=False)

In [3]:
model = GenerateModel()
model.load_state_dict(torch.load(model_path,weights_only=True,map_location=torch.device('cpu')))

<All keys matched successfully>

In [4]:
def obtain_weights(model):
    values = torch.Tensor([])
    for k, v in model.state_dict().items():
        values = torch.concat((values,v.flatten()),dim=0)
    return values
values = obtain_weights(model)

In [14]:
# plt.figure(figsize=(10,7))
# plt.hist(obtain_weights(GenerateModel()),bins = np.linspace(-1,1,num = 100))
# plt.hist(values,bins = np.linspace(-1,1,num = 100),alpha = 0.5)
# plt.show()

In [5]:
model.to(device)

ConvAttnPool(
  (embed): Embedding(150854, 100, padding_idx=150853)
  (conv): Conv1d(100, 15, kernel_size=(5,), stride=(1,), padding=(2,))
  (U): Linear(in_features=15, out_features=50, bias=True)
  (final): Linear(in_features=15, out_features=50, bias=True)
  (embed_drop): Dropout(p=0.2, inplace=False)
)

In [6]:
benchmark = {
    'auc_macro' : 0.884,
    'auc_micro' : 0.916,
    'f1_macro'  : 0.576,
    'f1_micro'  : 0.633
}

In [7]:
results = eval_model(model=model,
           device = device,
           data_loader=test_loader,
           label_space=50)

In [9]:
for k,v in benchmark.items():
    if k in results:
        print(compare_metric(benchmark,results,k))

auc_macro: 2.79% change (0.8840 → 0.9087)
auc_micro: 1.86% change (0.9160 → 0.9330)
f1_macro: 7.39% change (0.5760 → 0.6186)
f1_micro: 6.83% change (0.6330 → 0.6762)
